In [ ]:
# === Setup (works in Colab and locally) ===
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/jlattanzi4/energy-demand-forecasting.git /content/repo 2>/dev/null || true
    os.chdir('/content/repo')
    !pip install -q -e . prophet statsmodels
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

print(f'Working dir: {os.getcwd()}')

# Energy Demand Forecasting — Model Training & Comparison

| Model | Type | Description |
|-------|------|-------------|
| **Seasonal Naive** | Baseline | Same hour from previous week |
| **Prophet** | Statistical | Additive model with seasonality decomposition |
| **LSTM** | Deep Learning | 2-layer stacked LSTM (PyTorch) |
| **TCN** | Deep Learning | Dilated causal convolutions (PyTorch) |

**Forecast task**: Given 168 hours (1 week) of history, predict the next 24 hours.  
**Metrics**: MAE (MW), RMSE (MW), MAPE (%)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.train import prepare_data, create_dataloaders, train_model, EnergyDataset
from src.data.features import FEATURE_COLUMNS
from src.models.lstm import EnergyLSTM
from src.models.tcn import EnergyTCN
from src.evaluate import (
    evaluate_pytorch_model,
    evaluate_seasonal_naive,
    print_comparison_table,
    plot_predictions,
    plot_training_history,
)

device = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Using device: {device}')

%matplotlib inline

## 1. Prepare Data

In [ ]:
train_df, val_df, test_df, scaler = prepare_data('data/PJME_hourly.csv')

In [ ]:
WINDOW_SIZE = 168
FORECAST_HORIZON = 24
BATCH_SIZE = 64
EPOCHS = 50
LR = 1e-3

INPUT_SIZE = len(FEATURE_COLUMNS)
print(f'Input features: {INPUT_SIZE}')

train_loader, val_loader, test_loader = create_dataloaders(
    train_df, val_df, test_df,
    feature_cols=FEATURE_COLUMNS,
    window_size=WINDOW_SIZE,
    forecast_horizon=FORECAST_HORIZON,
    batch_size=BATCH_SIZE,
)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches:   {len(val_loader)}')
print(f'Test batches:  {len(test_loader)}')

## 2. Train LSTM

In [ ]:
lstm_model = EnergyLSTM(
    input_size=INPUT_SIZE,
    hidden_size=128,
    num_layers=2,
    dropout=0.2,
    forecast_horizon=FORECAST_HORIZON,
)

print(f'LSTM parameters: {sum(p.numel() for p in lstm_model.parameters()):,}')

lstm_model, lstm_history = train_model(
    lstm_model, train_loader, val_loader,
    epochs=EPOCHS, lr=LR, model_name='lstm',
    save_dir='models',
)

In [ ]:
plot_training_history(lstm_history, 'LSTM', save_path='models/lstm_training.png')

## 3. Train TCN

In [ ]:
tcn_model = EnergyTCN(
    input_size=INPUT_SIZE,
    num_channels=[64, 64, 64, 64],
    kernel_size=3,
    dropout=0.2,
    forecast_horizon=FORECAST_HORIZON,
)

print(f'TCN parameters: {sum(p.numel() for p in tcn_model.parameters()):,}')

tcn_model, tcn_history = train_model(
    tcn_model, train_loader, val_loader,
    epochs=EPOCHS, lr=LR, model_name='tcn',
    save_dir='models',
)

In [ ]:
plot_training_history(tcn_history, 'TCN', save_path='models/tcn_training.png')

## 4. Baseline: Seasonal Naive

In [ ]:
from src.data.preprocess import load_data, temporal_split

df_raw = load_data('data/PJME_hourly.csv')
_, _, test_raw = temporal_split(df_raw)
test_mw = test_raw['consumption_mw'].values

naive_results = evaluate_seasonal_naive(test_mw, season_length=168, horizon=24)
print(f"Seasonal Naive — MAE: {naive_results['mae']:.2f} MW | RMSE: {naive_results['rmse']:.2f} MW | MAPE: {naive_results['mape']:.2f}%")

## 5. Baseline: Prophet

In [ ]:
from src.models.baselines import ProphetBaseline
from src.evaluate import mae, rmse, mape

train_raw, _, test_raw = temporal_split(df_raw)

prophet_model = ProphetBaseline()
prophet_model.fit(train_raw)

test_hours = len(test_raw)
prophet_forecast = prophet_model.predict(periods=test_hours, freq='h')

prophet_preds = prophet_forecast['yhat'].values
prophet_actuals = test_raw['consumption_mw'].values[:len(prophet_preds)]

prophet_results = {
    'mae': mae(prophet_actuals, prophet_preds),
    'rmse': rmse(prophet_actuals, prophet_preds),
    'mape': mape(prophet_actuals, prophet_preds),
    'predictions': prophet_preds,
    'actuals': prophet_actuals,
}

print(f"Prophet — MAE: {prophet_results['mae']:.2f} MW | RMSE: {prophet_results['rmse']:.2f} MW | MAPE: {prophet_results['mape']:.2f}%")

## 6. Evaluate Deep Learning Models on Test Set

In [ ]:
lstm_results = evaluate_pytorch_model(lstm_model, test_loader, scaler, device)
tcn_results = evaluate_pytorch_model(tcn_model, test_loader, scaler, device)

print(f"LSTM — MAE: {lstm_results['mae']:.2f} MW | RMSE: {lstm_results['rmse']:.2f} MW | MAPE: {lstm_results['mape']:.2f}%")
print(f"TCN  — MAE: {tcn_results['mae']:.2f} MW | RMSE: {tcn_results['rmse']:.2f} MW | MAPE: {tcn_results['mape']:.2f}%")

## 7. Model Comparison

In [ ]:
all_results = {
    'Seasonal Naive': naive_results,
    'Prophet': prophet_results,
    'LSTM': lstm_results,
    'TCN': tcn_results,
}

print_comparison_table(all_results)

In [ ]:
model_names = list(all_results.keys())
mae_vals = [all_results[m]['mae'] for m in model_names]
rmse_vals = [all_results[m]['rmse'] for m in model_names]

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, mae_vals, width, label='MAE (MW)', color='#2196F3')
bars2 = ax.bar(x + width/2, rmse_vals, width, label='RMSE (MW)', color='#FF5722')

ax.set_ylabel('Error (MW)')
ax.set_title('Model Comparison — Test Set Performance')
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
            f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,
            f'{bar.get_height():.0f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('models/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Actual vs Predicted

In [ ]:
dl_results = {'LSTM': lstm_results, 'TCN': tcn_results}
plot_predictions(dl_results, sample_size=168, save_path='models/predictions_sample.png')

from IPython.display import Image
Image(filename='models/predictions_sample.png')

## 9. Sample Inference

In [ ]:
from src.predict import predict

df_full = load_data('data/PJME_hourly.csv')
recent = df_full.tail(336)

best_model = lstm_model
forecast = predict(best_model, recent, scaler)

last_ts = recent.index[-1]
forecast_times = pd.date_range(start=last_ts + pd.Timedelta(hours=1), periods=24, freq='h')

forecast_df = pd.DataFrame({'timestamp': forecast_times, 'predicted_mw': forecast})
print('24-Hour Forecast:')
print(forecast_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(recent.index[-48:], recent['consumption_mw'].values[-48:], label='Recent (48h)', color='#2196F3')
ax.plot(forecast_times, forecast, label='Forecast (24h)', color='#FF5722', linewidth=2, marker='o', markersize=4)
ax.axvline(last_ts, color='gray', linestyle='--', alpha=0.5, label='Forecast start')
ax.set_xlabel('Time')
ax.set_ylabel('Energy (MW)')
ax.set_title('Sample 24-Hour Forecast')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

| Model | MAE (MW) | RMSE (MW) | MAPE (%) |
|-------|----------|-----------|----------|
| Seasonal Naive | — | — | — |
| Prophet | — | — | — |
| **LSTM** | — | — | — |
| **TCN** | — | — | — |

*(Fill in after training)*

### Next Steps
- Add weather covariates (temperature from Open-Meteo API)
- Experiment with Temporal Fusion Transformer (TFT)
- Set up automated retraining pipeline
- Add data drift monitoring in production